# Filter-bank estimate of `rhof_veff_ratio`

Estimate one `rhof_veff_ratio_hat` per tracking case from the full LOS-removed carrier phase and tracked intensity time series. Truth is read from `diagnostics.csv`.

In [1]:
from pathlib import Path
import json

import kagglehub
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plotly_config = {
    "scrollZoom": True,
    "displaylogo": False,
    "responsive": True,
}


/home/rodrigo/github/real-time-bp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## General Configuration

In [2]:
DATASET_HANDLE = "rodrigodlf/backpropagation/versions/16"
USE_LOCAL_OBSERVABLES = False
LOCAL_OBSERVABLES_DIR = Path("../tracking-loops-dataset-gen/data/observables/main")

rho_grid = np.linspace(0.5, 10.0, 250)
UNWRAPPED_PHASE_DIVERGENCE_THRESHOLD_CYCLES = 10.0
USE_PHASE_ALIGNMENT = False
SCREEN_PHASE_MASK_THRESHOLD_DB = -15.0
SCREEN_PHASE_POST_FADE_MASK_SEC = 2.0
SLIP_DETECT_WINDOW_SEC = 2.0
SLIP_INTEGER_TOLERANCE_CYCLES = 0.20

track_keys = ["city", "svid", "frequency", "severity"]
true_refractive_phase_col = "true_refractive_phase_m"
backprop_phase_label = "LOS-removed carrier phase"
refractive_truth_label = "direct true refractive phase"

SPEED_OF_LIGHT = 299_792_458.0
GPS_CARRIER_FREQUENCY_HZ = {"L1": 1_575.42e6}
GPS_CARRIER_WAVELENGTH_M = {
    freq: SPEED_OF_LIGHT / freq_hz
    for freq, freq_hz in GPS_CARRIER_FREQUENCY_HZ.items()
}

## Dataset Download and Handling

In [3]:
if USE_LOCAL_OBSERVABLES:
    observables_dir = LOCAL_OBSERVABLES_DIR
else:
    observables_dir = Path(kagglehub.dataset_download(DATASET_HANDLE)) / "observables"

meta = json.loads((observables_dir / "meta.json").read_text())
diagnostics = pd.read_csv(observables_dir / meta["diagnostics"])

string_dtypes = {
    "city": "string",
    "svid": "string",
    "frequency": "string",
    "severity": "string",
}
observables = pd.concat(
    [
        pd.read_csv(observables_dir / part["path"], dtype=string_dtypes)
        for part in meta["partitions"]
    ],
    ignore_index=True,
)
observables["tracked_scint_phase_m"] = (
    observables["carrier_phase_est_m"] - observables["los_phase_m"]
)
observables["carrier_wavelength_m"] = (
    observables["frequency"].map(GPS_CARRIER_WAVELENGTH_M).astype("float64")
)
observables = observables.sort_values(track_keys + ["epoch_idx"])
case_groups = {
    key: frame
    for key, frame in observables.groupby(track_keys, sort=False, observed=True)
}

print("observables_dir:", observables_dir)
print("backpropagation phase input:", backprop_phase_label)
print("refractive phase reference:", refractive_truth_label)
print("tracking cases:", len(diagnostics))
print("observable rows:", len(observables))
diagnostics.head()

observables_dir: /home/rodrigo/.cache/kagglehub/datasets/rodrigodlf/backpropagation/versions/16/observables
backpropagation phase input: LOS-removed carrier phase
refractive phase reference: direct true refractive phase
tracking cases: 33
observable rows: 1435500


,tracking_case_id,city,svid,frequency,severity,rhof_veff_ratio,bypass_dll,enable_doppler_aiding,acquisition_enabled,acq_success,acq_metric,acq_code_phase_chips,acq_doppler_hz,acq_doppler_error_hz,noise_seed,loss_of_lock
0,1,palmas,G01,L1,strong,1.676684,True,False,True,True,NaN,NaN,NaN,NaN,1639030180,False
1,2,palmas,G03,L1,strong,5.981997,True,False,True,True,NaN,NaN,NaN,NaN,830043460,False
2,3,palmas,G08,L1,strong,1.641801,True,False,True,True,NaN,NaN,NaN,NaN,2234450039,False
3,4,palmas,G11,L1,weak,1.259302,True,False,True,True,NaN,NaN,NaN,NaN,131598802,False
4,5,palmas,G14,L1,strong,5.860273,True,False,True,True,NaN,NaN,NaN,NaN,157149062,False


In [4]:
def phase_m_to_rad(track, column):
    return (
        2.0
        * np.pi
        * track[column].to_numpy(dtype=float)
        / track["carrier_wavelength_m"].to_numpy(dtype=float)
    )


def field_from_track(track, intensity_col, phase_col):
    phase_rad = phase_m_to_rad(track, phase_col)
    amplitude = np.sqrt(
        np.clip(track[intensity_col].to_numpy(dtype=float), 0.0, None)
    )
    return amplitude * np.exp(1j * phase_rad), phase_rad


def tracked_field(track):
    return field_from_track(track, "tracked_intensity", "tracked_scint_phase_m")


def truth_field(track):
    return field_from_track(track, "true_intensity", "scint_phase_m")


def carrier_phase_diagnostics(track):
    wavelength = track["carrier_wavelength_m"].to_numpy(dtype=float)
    tracked_cycles = track["tracked_scint_phase_m"].to_numpy(dtype=float) / wavelength
    truth_cycles = track["scint_phase_m"].to_numpy(dtype=float) / wavelength
    error_cycles = tracked_cycles - truth_cycles
    unwrapped_error = error_cycles - error_cycles[0]
    wrapped_error = ((error_cycles + 0.5) % 1.0) - 0.5
    return tracked_cycles, truth_cycles, unwrapped_error, wrapped_error


def backpropagate_fft(u, dt, rho):
    f = np.fft.fftfreq(u.size, d=dt)
    H_inv = np.exp(0.5j * (2.0 * np.pi * f * rho) ** 2)
    return np.fft.ifft(np.fft.fft(u) * H_inv)


def forward_propagate_fft(psi, dt, rho):
    f = np.fft.fftfreq(psi.size, d=dt)
    H_fwd = np.exp(-0.5j * (2.0 * np.pi * f * rho) ** 2)
    return np.fft.ifft(np.fft.fft(psi) * H_fwd)


def power_db(power):
    return 10.0 * np.log10(np.maximum(power, np.finfo(float).tiny))


def field_intensity_db(field):
    return power_db(np.abs(field) ** 2)


def continuous_field_phase(field):
    return np.unwrap(np.angle(field))


def align_start_cycle(phase_rad, reference_phase_rad):
    cycle_shift = 2.0 * np.pi * np.round(
        (reference_phase_rad[0] - phase_rad[0]) / (2.0 * np.pi)
    )
    return phase_rad + cycle_shift


def maybe_align_phase(phase_rad, reference_phase_rad):
    if USE_PHASE_ALIGNMENT:
        return align_start_cycle(phase_rad, reference_phase_rad)
    return phase_rad


def phase_error_cycles(phase_rad, reference_phase_rad, center=None):
    error = (phase_rad - reference_phase_rad) / (2.0 * np.pi)
    if center is None:
        return error
    if center == "first":
        return error - error[0]
    if center == "mean":
        return error - np.mean(error)
    if center == "median":
        return error - np.median(error)
    raise ValueError("center must be one of None, 'first', 'mean', or 'median'.")


def interpolate_masked_samples(x, values, reliable_mask):
    x = np.asarray(x, dtype=float)
    values = np.asarray(values, dtype=float)
    reliable_mask = np.asarray(reliable_mask, dtype=bool)
    if x.shape != values.shape or values.shape != reliable_mask.shape:
        raise ValueError("x, values, and reliable_mask must have the same shape.")
    if np.count_nonzero(reliable_mask) < 2:
        return values.copy()
    filled = values.copy()
    filled[~reliable_mask] = np.interp(
        x[~reliable_mask],
        x[reliable_mask],
        values[reliable_mask],
    )
    return filled


def wrapped_phase_error_cycles(phase_rad, reference_phase_rad):
    return np.angle(np.exp(1j * (phase_rad - reference_phase_rad))) / (2.0 * np.pi)


def unreliable_runs(reliable_mask):
    unreliable = ~np.asarray(reliable_mask, dtype=bool)
    padded = np.concatenate(([False], unreliable, [False]))
    transitions = np.diff(padded.astype(int))
    starts = np.flatnonzero(transitions == 1)
    stops = np.flatnonzero(transitions == -1)
    return list(zip(starts, stops))


def extend_mask_after_unreliable_runs(t, reliable_mask, post_fade_sec):
    t = np.asarray(t, dtype=float)
    extended = np.asarray(reliable_mask, dtype=bool).copy()
    for _, stop in unreliable_runs(reliable_mask):
        fade_end = t[stop - 1]
        post_fade = (t > fade_end) & (t <= fade_end + float(post_fade_sec))
        extended[post_fade] = False
    return extended


def detect_post_fade_cycle_slips(
    t,
    fade_reliable_mask,
    tracked_phase_rad,
    predicted_phase_rad,
    intensity_db,
    *,
    pre_window_sec,
    post_window_sec,
    integer_tolerance_cycles,
):
    t = np.asarray(t, dtype=float)
    residual = phase_error_cycles(tracked_phase_rad, predicted_phase_rad)
    intensity_db = np.asarray(intensity_db, dtype=float)
    rows = []
    for start, stop in unreliable_runs(fade_reliable_mask):
        fade_start = t[start]
        fade_end = t[stop - 1]
        pre = (t >= fade_start - pre_window_sec) & (t < fade_start) & fade_reliable_mask
        post = (t > fade_end) & (t <= fade_end + post_window_sec)
        if np.count_nonzero(pre) < 3 or np.count_nonzero(post) < 3:
            continue
        pre_median = float(np.median(residual[pre]))
        post_median = float(np.median(residual[post]))
        step_cycles = post_median - pre_median
        nearest_integer = float(np.round(step_cycles))
        integer_distance = float(abs(step_cycles - nearest_integer))
        rows.append(
            {
                "fade_start_s": float(fade_start),
                "fade_end_s": float(fade_end),
                "fade_duration_s": float(fade_end - fade_start),
                "min_intensity_db": float(np.min(intensity_db[start:stop])),
                "pre_residual_cycles": pre_median,
                "post_residual_cycles": post_median,
                "step_cycles": float(step_cycles),
                "nearest_integer_cycles": nearest_integer,
                "integer_distance_cycles": integer_distance,
                "cycle_slip_candidate": bool(
                    abs(nearest_integer) >= 1.0
                    and integer_distance <= integer_tolerance_cycles
                ),
            }
        )
    return pd.DataFrame(
        rows,
        columns=[
            "fade_start_s",
            "fade_end_s",
            "fade_duration_s",
            "min_intensity_db",
            "pre_residual_cycles",
            "post_residual_cycles",
            "step_cycles",
            "nearest_integer_cycles",
            "integer_distance_cycles",
            "cycle_slip_candidate",
        ],
    )


def rms(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values**2)))


def estimate_rho(track, rho_grid):
    u, _ = tracked_field(track)
    t = track["time_sec"].to_numpy(dtype=float)
    dt = float(np.median(np.diff(t)))
    f = np.fft.fftfreq(u.size, d=dt)
    U = np.fft.fft(u)

    scores = np.empty(rho_grid.size, dtype=float)
    for idx, rho in enumerate(rho_grid):
        H_inv = np.exp(0.5j * (2.0 * np.pi * f * rho) ** 2)
        scores[idx] = np.var(np.abs(np.fft.ifft(U * H_inv)))

    best_idx = int(np.argmin(scores))
    return float(rho_grid[best_idx]), float(scores[best_idx]), scores

In [5]:
estimate_rows = []
score_by_case = {}
case_columns = [
    "tracking_case_id",
    "city",
    "svid",
    "frequency",
    "severity",
    "rhof_veff_ratio",
    "bypass_dll",
    "loss_of_lock",
]

for diag in diagnostics.sort_values("tracking_case_id").to_dict("records"):
    key = tuple(diag[col] for col in track_keys)
    track = case_groups[key]
    rho_hat, score_min, scores = estimate_rho(track, rho_grid)
    _, _, phase_error, wrapped_phase_error = carrier_phase_diagnostics(track)
    phase_error_max = float(np.max(np.abs(phase_error)))
    case_id = int(diag["tracking_case_id"])

    row = {col: diag[col] for col in case_columns}
    row.update(
        tracking_case_id=case_id,
        rhof_veff_ratio=float(diag["rhof_veff_ratio"]),
        rhof_veff_ratio_hat=rho_hat,
        score_min=score_min,
        bypass_dll=bool(diag["bypass_dll"]),
        loss_of_lock=bool(diag["loss_of_lock"]),
        carrier_phase_error_max_abs_cycles=phase_error_max,
        carrier_phase_error_final_cycles=float(phase_error[-1]),
        wrapped_phase_error_rms_cycles=float(np.sqrt(np.mean(wrapped_phase_error**2))),
        unwrapped_phase_diverged=bool(
            phase_error_max > UNWRAPPED_PHASE_DIVERGENCE_THRESHOLD_CYCLES
        ),
    )
    estimate_rows.append(row)
    score_by_case[case_id] = scores

estimates = pd.DataFrame(estimate_rows)
estimates

,tracking_case_id,city,svid,frequency,severity,rhof_veff_ratio,bypass_dll,loss_of_lock,rhof_veff_ratio_hat,score_min,carrier_phase_error_max_abs_cycles,carrier_phase_error_final_cycles,wrapped_phase_error_rms_cycles,unwrapped_phase_diverged
0,1,palmas,G01,L1,strong,1.676684,True,False,1.682731,0.006809,2.355833,-1.994319,0.018513,False
1,2,palmas,G03,L1,strong,5.981997,True,False,5.955823,0.016365,0.283843,0.007069,0.014651,False
2,3,palmas,G08,L1,strong,1.641801,True,False,1.644578,0.010509,2.331693,2.022515,0.018712,False
3,4,palmas,G11,L1,weak,1.259302,True,False,1.263052,0.005097,0.065736,0.018506,0.012737,False
4,5,palmas,G14,L1,strong,5.860273,True,False,5.841365,0.022388,0.258162,-0.011144,0.015023,False
5,6,palmas,G22,L1,weak,3.183382,True,False,3.170683,0.004619,0.053620,0.024733,0.009776,False
6,7,palmas,G23,L1,weak,1.627129,True,False,1.644578,0.007237,0.054661,-0.008661,0.011982,False
7,8,palmas,G27,L1,weak,7.924686,True,False,0.538153,0.008769,0.044900,-0.004358,0.009739,False
8,9,palmas,G31,L1,weak,1.064834,True,False,1.072289,0.005578,0.064073,-0.013421,0.013299,False
9,10,presidente_prudente,G01,L1,strong,1.676684,True,True,1.682731,0.009997,2.267243,-2.002076,0.020085,False


In [6]:
loss_summary_parts = [
    pd.DataFrame(
        {
            "group": ["overall"],
            "value": ["all"],
            "tracking_cases": [len(estimates)],
            "loss_of_lock_cases": [int(estimates["loss_of_lock"].sum())],
        }
    )
]

for group_col in ["severity", "city", "frequency"]:
    part = (
        estimates.groupby(group_col, observed=True)["loss_of_lock"]
        .agg(tracking_cases="size", loss_of_lock_cases="sum")
        .reset_index()
        .rename(columns={group_col: "value"})
    )
    part.insert(0, "group", group_col)
    loss_summary_parts.append(part)

loss_of_lock_summary = pd.concat(loss_summary_parts, ignore_index=True)
loss_of_lock_summary["loss_of_lock_cases"] = loss_of_lock_summary[
    "loss_of_lock_cases"
].astype(int)
loss_of_lock_summary["lock_retained_cases"] = (
    loss_of_lock_summary["tracking_cases"]
    - loss_of_lock_summary["loss_of_lock_cases"]
)
loss_of_lock_summary["loss_of_lock_rate_pct"] = (
    100.0
    * loss_of_lock_summary["loss_of_lock_cases"]
    / loss_of_lock_summary["tracking_cases"]
).round(2)
loss_of_lock_summary

,group,value,tracking_cases,loss_of_lock_cases,lock_retained_cases,loss_of_lock_rate_pct
0,overall,all,33,6,27,18.18
1,severity,strong,16,6,10,37.50
2,severity,weak,17,0,17,0.00
3,city,palmas,9,0,9,0.00
4,city,porto_alegre,8,1,7,12.50
5,city,presidente_prudente,8,3,5,37.50
6,city,sao_jose_dos_campos,8,2,6,25.00
7,frequency,L1,33,6,27,18.18


In [7]:
truth = estimates["rhof_veff_ratio"].to_numpy(dtype=float)
estimated = estimates["rhof_veff_ratio_hat"].to_numpy(dtype=float)
error = estimated - truth

metrics = pd.Series(
    {
        "RMSE": np.sqrt(np.mean(error**2)),
        "MAPE [%]": 100.0 * np.mean(np.abs(error / truth)),
        "MAE": np.mean(np.abs(error)),
        "R^2": 1.0
        - np.sum(error**2) / np.sum((truth - np.mean(truth)) ** 2),
    }
)
metrics

RMSE        1.286905
MAPE [%]    3.408280
MAE         0.243044
R^2         0.614023
dtype: float64

In [8]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=truth,
        y=estimated,
        mode="markers",
        name="Cases",
        customdata=estimates[["tracking_case_id", "city", "svid", "severity"]],
        marker={"size": 9, "opacity": 0.85},
        hovertemplate=(
            "case %{customdata[0]}<br>"
            "%{customdata[1]} %{customdata[2]} (%{customdata[3]})<br>"
            "truth=%{x:.3f} s<br>estimated=%{y:.3f} s<extra></extra>"
        ),
    )
)

lo = min(truth.min(), estimated.min())
hi = max(truth.max(), estimated.max())
pad = 0.05 * (hi - lo)
line = np.array([lo - pad, hi + pad])
fig.add_trace(
    go.Scatter(
        x=line,
        y=line,
        mode="lines",
        name="Perfect prediction",
        line={"color": "black", "dash": "dash", "width": 1.2},
        hoverinfo="skip",
    )
)

fig.update_layout(
    title="Truth vs estimated rhof_veff_ratio",
    template="plotly_white",
    width=520,
    height=520,
    margin={"l": 70, "r": 30, "t": 70, "b": 60},
    xaxis_title="Truth rhof_veff_ratio [s]",
    yaxis_title="Estimated rhof_veff_ratio_hat [s]",
    legend={"x": 0.02, "y": 0.98},
)
fig.update_xaxes(range=line, showgrid=True, zeroline=False)
fig.update_yaxes(range=line, showgrid=True, zeroline=False, scaleanchor="x", scaleratio=1)
fig.show(config=plotly_config)


In [9]:
example_idx = 12
example = estimates.iloc[example_idx - 1]
example_track = case_groups[tuple(example[col] for col in track_keys)]
rho = float(example["rhof_veff_ratio"])
rho_hat = float(example["rhof_veff_ratio_hat"])

u_tracked, tracked_phase_rad = tracked_field(example_track)
u_truth, truth_phase_rad = truth_field(example_track)
t_example = example_track["time_sec"].to_numpy(dtype=float)
dt_example = float(np.median(np.diff(t_example)))
tracked_intensity_db = power_db(example_track["tracked_intensity"].to_numpy(dtype=float))
truth_intensity_db = power_db(example_track["true_intensity"].to_numpy(dtype=float))
screen_phase_mask = tracked_intensity_db >= SCREEN_PHASE_MASK_THRESHOLD_DB
slip_detection_mask = extend_mask_after_unreliable_runs(
    t_example,
    screen_phase_mask,
    SCREEN_PHASE_POST_FADE_MASK_SEC,
)
screen_masked_fraction = float(np.mean(~screen_phase_mask))
slip_detection_masked_fraction = float(np.mean(~slip_detection_mask))

psi_bp = backpropagate_fft(u_tracked, dt_example, rho)
psi_hat = backpropagate_fft(u_tracked, dt_example, rho_hat)
scores_example = score_by_case[int(example["tracking_case_id"])]

reference_phase = phase_m_to_rad(example_track, true_refractive_phase_col)
bp_phase = maybe_align_phase(continuous_field_phase(psi_bp), reference_phase)
bp_hat_phase = maybe_align_phase(continuous_field_phase(psi_hat), reference_phase)
bp_phase_masked = interpolate_masked_samples(t_example, bp_phase, screen_phase_mask)
bp_hat_phase_masked = interpolate_masked_samples(t_example, bp_hat_phase, screen_phase_mask)
bp_phase_detection = interpolate_masked_samples(t_example, bp_phase, slip_detection_mask)

psi_bp_masked = np.abs(psi_bp) * np.exp(1j * bp_phase_masked)
psi_hat_masked = np.abs(psi_hat) * np.exp(1j * bp_hat_phase_masked)
psi_bp_detection = np.abs(psi_bp) * np.exp(1j * bp_phase_detection)
u_forward = forward_propagate_fft(psi_bp, dt_example, rho)
u_forward_masked = forward_propagate_fft(psi_bp_masked, dt_example, rho)
u_forward_detection = forward_propagate_fft(psi_bp_detection, dt_example, rho)
u_forward_hat = forward_propagate_fft(psi_hat, dt_example, rho_hat)
u_forward_hat_masked = forward_propagate_fft(psi_hat_masked, dt_example, rho_hat)

forward_phase = maybe_align_phase(continuous_field_phase(u_forward), tracked_phase_rad)
forward_masked_phase = maybe_align_phase(continuous_field_phase(u_forward_masked), tracked_phase_rad)
forward_detection_phase = maybe_align_phase(continuous_field_phase(u_forward_detection), tracked_phase_rad)
forward_hat_phase = maybe_align_phase(continuous_field_phase(u_forward_hat), tracked_phase_rad)
forward_hat_masked_phase = maybe_align_phase(
    continuous_field_phase(u_forward_hat_masked),
    tracked_phase_rad,
)
tracked_phase_plot = maybe_align_phase(tracked_phase_rad, bp_phase)
truth_phase_plot = maybe_align_phase(truth_phase_rad, tracked_phase_plot)

carrier_phase_error_cycles = phase_error_cycles(tracked_phase_rad, truth_phase_rad, center="first")
carrier_forward_phase_error_cycles = phase_error_cycles(forward_phase, truth_phase_rad, center="first")
carrier_forward_masked_phase_error_cycles = phase_error_cycles(
    forward_masked_phase,
    truth_phase_rad,
    center="first",
)
carrier_forward_detection_phase_error_cycles = phase_error_cycles(
    forward_detection_phase,
    truth_phase_rad,
    center="first",
)
carrier_tracked_forward_detection_error_cycles = phase_error_cycles(
    tracked_phase_rad,
    forward_detection_phase,
    center="first",
)

lock_status = "detector lost lock" if bool(example["loss_of_lock"]) else "detector retained lock"
phase_status = (
    "unwrapped phase diverged"
    if bool(example["unwrapped_phase_diverged"])
    else "unwrapped phase nominal"
)
alignment_status = "phase alignment on" if USE_PHASE_ALIGNMENT else "phase alignment off"
figure_title = (
    f"Example {example_idx} | tracking case {int(example['tracking_case_id'])}: "
    f"{example['city']} {example['svid']} {example['frequency']} "
    f"({example['severity']}, {lock_status}, {phase_status}, {alignment_status}) | "
    f"input: {backprop_phase_label} | reference: {refractive_truth_label}"
)
mask_label = f"intensity mask ({SCREEN_PHASE_MASK_THRESHOLD_DB:g} dB)"

fig = make_subplots(
    rows=4,
    cols=2,
    specs=[
        [{}, {}],
        [{}, {}],
        [{"colspan": 2}, None],
        [{}, {}],
    ],
    subplot_titles=(
        "Backpropagation and no-intermediate forward phase",
        "Refractive phase retrieval",
        "Filter-bank score",
        "Backpropagated intensity magnitude",
        "Tracked, truth, and forward-propagated signal intensity",
        "Carrier phase errors",
        "Refractive phase reconstruction error",
    ),
    vertical_spacing=0.09,
    horizontal_spacing=0.08,
)


def add_line(
    row,
    col,
    x,
    y,
    name,
    *,
    legend_id,
    dash=None,
    color=None,
    opacity=None,
    showlegend=True,
):
    line = {}
    if dash is not None:
        line["dash"] = dash
    if color is not None:
        line["color"] = color
    fig.add_trace(
        go.Scattergl(
            x=x,
            y=y,
            mode="lines",
            name=name,
            legend=legend_id,
            line=line,
            opacity=opacity,
            showlegend=showlegend,
        ),
        row=row,
        col=col,
    )


def add_zero_line(row, col, legend_id):
    add_line(
        row,
        col,
        [float(t_example[0]), float(t_example[-1])],
        [0.0, 0.0],
        "Zero reference",
        legend_id=legend_id,
        color="black",
        opacity=0.6,
        showlegend=False,
    )


add_line(1, 1, t_example, bp_phase, "Backpropagated phase", legend_id="legend")
add_line(1, 1, t_example, tracked_phase_plot, f"Input: {backprop_phase_label}", legend_id="legend")
add_line(1, 1, t_example, forward_phase, "Forward after backpropagation", legend_id="legend", dash="dash")
add_line(1, 1, t_example, forward_masked_phase, "Forward after masked screen", legend_id="legend", dash="dashdot")
add_line(1, 1, t_example, truth_phase_plot, "Truth scintillation phase", legend_id="legend", dash="dash")

add_line(1, 2, t_example, reference_phase, f"Reference: {refractive_truth_label}", legend_id="legend2")
add_line(1, 2, t_example, bp_phase, "Retrieved with truth rho", legend_id="legend2", opacity=0.8)
add_line(
    1,
    2,
    t_example,
    bp_phase_masked,
    f"Truth rho + {mask_label}",
    legend_id="legend2",
    dash="dashdot",
)
add_line(1, 2, t_example, bp_hat_phase, "Retrieved with estimated rho", legend_id="legend2", opacity=0.75)
add_line(
    1,
    2,
    t_example,
    bp_hat_phase_masked,
    f"Estimated rho + {mask_label}",
    legend_id="legend2",
    dash="dashdot",
    opacity=0.75,
)

score_limits = [float(np.min(scores_example)), float(np.max(scores_example))]
add_line(2, 1, rho_grid, scores_example, "Filter-bank score", legend_id="legend3")
fig.add_trace(
    go.Scatter(
        x=[rho, rho],
        y=score_limits,
        mode="lines",
        name="Truth",
        legend="legend3",
        line={"color": "green", "dash": "dash"},
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=[rho_hat, rho_hat],
        y=score_limits,
        mode="lines",
        name="Estimate",
        legend="legend3",
        line={"color": "red", "dash": "dot"},
    ),
    row=2,
    col=1,
)

add_line(2, 2, t_example, field_intensity_db(psi_bp), "Truth rho", legend_id="legend4")
add_line(2, 2, t_example, field_intensity_db(psi_hat), "Estimated rho", legend_id="legend4", opacity=0.8)

forward_intensity_db = field_intensity_db(u_forward)
forward_masked_intensity_db = field_intensity_db(u_forward_masked)
add_line(3, 1, t_example, tracked_intensity_db, "Tracked intensity", legend_id="legend5", opacity=0.85)
add_line(3, 1, t_example, truth_intensity_db, "Truth intensity", legend_id="legend5", opacity=0.85)
add_line(
    3,
    1,
    t_example,
    forward_intensity_db,
    "Forward after backpropagation",
    legend_id="legend5",
    dash="dash",
    opacity=0.85,
)
add_line(
    3,
    1,
    t_example,
    forward_masked_intensity_db,
    "Forward after masked screen",
    legend_id="legend5",
    dash="dashdot",
    opacity=0.85,
)
add_line(
    3,
    1,
    [float(t_example[0]), float(t_example[-1])],
    [SCREEN_PHASE_MASK_THRESHOLD_DB, SCREEN_PHASE_MASK_THRESHOLD_DB],
    "Mask threshold",
    legend_id="legend5",
    dash="dot",
    color="black",
    opacity=0.75,
)

add_line(4, 1, t_example, carrier_phase_error_cycles, "Tracked - truth", legend_id="legend6")
add_line(4, 1, t_example, carrier_forward_phase_error_cycles, "Forward - truth", legend_id="legend6")
add_line(
    4,
    1,
    t_example,
    carrier_forward_masked_phase_error_cycles,
    "Forward mask - truth",
    legend_id="legend6",
    dash="dashdot",
)
add_line(
    4,
    1,
    t_example,
    carrier_forward_detection_phase_error_cycles,
    "Forward held-out - truth",
    legend_id="legend6",
    dash="dot",
)
add_line(
    4,
    1,
    t_example,
    carrier_tracked_forward_detection_error_cycles,
    "Tracked - held-out forward",
    legend_id="legend6",
    dash="longdash",
)
add_zero_line(4, 1, "legend6")

add_line(
    4,
    2,
    t_example,
    phase_error_cycles(bp_phase, reference_phase),
    "Truth rho",
    legend_id="legend7",
)
add_line(
    4,
    2,
    t_example,
    phase_error_cycles(bp_phase_masked, reference_phase),
    "Truth rho + mask",
    legend_id="legend7",
    dash="dashdot",
)
add_line(
    4,
    2,
    t_example,
    phase_error_cycles(bp_hat_phase, reference_phase),
    "Estimated rho",
    legend_id="legend7",
    opacity=0.8,
)
add_line(
    4,
    2,
    t_example,
    phase_error_cycles(bp_hat_phase_masked, reference_phase),
    "Estimated rho + mask",
    legend_id="legend7",
    dash="dashdot",
    opacity=0.8,
)
add_zero_line(4, 2, "legend7")

axis_labels = {
    (1, 1): ("Time [s]", "Phase [rad]"),
    (1, 2): ("Time [s]", "Phase [rad]"),
    (2, 1): ("rhof_veff_ratio [s]", "Amplitude variance"),
    (2, 2): ("Time [s]", "Intensity [dB]"),
    (3, 1): ("Time [s]", "Intensity [dB]"),
    (4, 1): ("Time [s]", "Difference [cycles]"),
    (4, 2): ("Time [s]", "Difference [cycles]"),
}
for (row, col), (xlabel, ylabel) in axis_labels.items():
    fig.update_xaxes(title_text=xlabel, showgrid=True, row=row, col=col)
    fig.update_yaxes(title_text=ylabel, showgrid=True, row=row, col=col)

legend_common = {
    "x": 1.01,
    "xanchor": "left",
    "yanchor": "top",
    "bgcolor": "rgba(255,255,255,0.88)",
    "bordercolor": "rgba(0,0,0,0.18)",
    "borderwidth": 1,
    "font": {"size": 9},
}
fig.update_layout(
    title={"text": figure_title, "x": 0.5, "xanchor": "center"},
    template="plotly_white",
    width=1180,
    height=1100,
    margin={"l": 65, "r": 265, "t": 115, "b": 65},
    hovermode="closest",
    legend={**legend_common, "y": 1.00, "title": {"text": "Phase"}},
    legend2={**legend_common, "y": 0.84, "title": {"text": "Retrieval"}},
    legend3={**legend_common, "y": 0.62, "title": {"text": "Score"}},
    legend4={**legend_common, "y": 0.50, "title": {"text": "BP intensity"}},
    legend5={**legend_common, "y": 0.36, "title": {"text": "Signal intensity"}},
    legend6={**legend_common, "y": 0.18, "title": {"text": "Carrier error"}},
    legend7={**legend_common, "y": 0.05, "title": {"text": "Screen error"}},
)
fig.show(config=plotly_config)

def phase_screen_row(name, phase, forward_phase, forward_intensity_db, masked_fraction=0.0):
    refractive_error = phase_error_cycles(phase, reference_phase)
    return {
        "case": name,
        "masked_fraction": float(masked_fraction),
        "refractive_error_mean_cycles": float(np.mean(refractive_error)),
        "refractive_error_rms_cycles": rms(refractive_error),
        "refractive_error_median_centered_rms_cycles": rms(
            phase_error_cycles(phase, reference_phase, center="median")
        ),
        "forward_phase_error_rms_cycles": rms(
            phase_error_cycles(forward_phase, truth_phase_rad, center="first")
        ),
        "forward_wrapped_phase_error_rms_cycles": rms(
            wrapped_phase_error_cycles(forward_phase, truth_phase_rad)
        ),
        "forward_intensity_rmse_db": rms(forward_intensity_db - truth_intensity_db),
    }


phase_screen_results = pd.DataFrame(
    [
        phase_screen_row("truth rho", bp_phase, forward_phase, forward_intensity_db),
        phase_screen_row(
            f"truth rho + {mask_label}",
            bp_phase_masked,
            forward_masked_phase,
            forward_masked_intensity_db,
            screen_masked_fraction,
        ),
        phase_screen_row(
            "estimated rho",
            bp_hat_phase,
            forward_hat_phase,
            field_intensity_db(u_forward_hat),
        ),
        phase_screen_row(
            f"estimated rho + {mask_label}",
            bp_hat_phase_masked,
            forward_hat_masked_phase,
            field_intensity_db(u_forward_hat_masked),
            screen_masked_fraction,
        ),
    ]
)
slip_candidates = detect_post_fade_cycle_slips(
    t_example,
    screen_phase_mask,
    tracked_phase_rad,
    forward_detection_phase,
    tracked_intensity_db,
    pre_window_sec=SLIP_DETECT_WINDOW_SEC,
    post_window_sec=SLIP_DETECT_WINDOW_SEC,
    integer_tolerance_cycles=SLIP_INTEGER_TOLERANCE_CYCLES,
)
phase_screen_results


,case,masked_fraction,refractive_error_mean_cycles,refractive_error_rms_cycles,refractive_error_median_centered_rms_cycles,forward_phase_error_rms_cycles,forward_wrapped_phase_error_rms_cycles,forward_intensity_rmse_db
0,truth rho,0.000000,0.572867,0.573052,0.014583,0.754699,0.021560,1.104668
1,truth rho + intensity mask (-15 dB),0.010782,0.572884,0.573069,0.014566,0.755216,0.021571,1.092953
2,estimated rho,0.000000,0.573800,0.573986,0.014636,0.754699,0.021560,1.104668
3,estimated rho + intensity mask (-15 dB),0.010782,0.573820,0.574006,0.014631,0.754872,0.021730,1.106396


In [10]:
slip_candidates


,fade_start_s,fade_end_s,fade_duration_s,min_intensity_db,pre_residual_cycles,post_residual_cycles,step_cycles,nearest_integer_cycles,integer_distance_cycles,cycle_slip_candidate
0,39.56,39.94,0.38,-23.132506,0.001571,0.000424,-0.001147,-0.0,0.001147,False
1,43.24,43.64,0.40,-26.419641,-0.025762,-0.035528,-0.009766,-0.0,0.009766,False
2,46.40,46.42,0.02,-15.355920,-0.031198,-0.019992,0.011206,0.0,0.011206,False
3,131.16,131.18,0.02,-18.048407,-0.015320,0.017385,0.032705,0.0,0.032705,False
4,131.24,131.38,0.14,-22.725279,-0.014320,0.017385,0.031706,0.0,0.031706,False
...,...,...,...,...,...,...,...,...,...,...
97,769.52,769.54,0.02,-30.376436,-1.981212,-1.985812,-0.004600,-0.0,0.004600,False
98,772.44,772.46,0.02,-18.415468,-1.986443,-2.026578,-0.040135,-0.0,0.040135,False
99,836.68,836.78,0.10,-20.250160,-1.999219,-2.001438,-0.002219,-0.0,0.002219,False
100,840.64,840.66,0.02,-15.644948,-2.000795,-2.003306,-0.002511,-0.0,0.002511,False
